# Which model do we buy? Forests and boosting on real bookings

CL3 of *Predictive Analytics*. Work in **pairs**; swap at every station: **predict, run, change, decide.**

**Read** questions are in the style of the exam: answer them on paper first, then run the cell below.

**Stuck?** Every station only needs the *Setup* cell. Just go on with the next station.

| Station | Question | Time |
|---|---|---|
| 1 | Many trees | 10 min |
| 2 | What does the forest look at? | 10 min |
| 3 | Boosting: how many rounds? | 12 min |
| 4 | Promise and test | 10 min |
| Extra | For the fast ones | |

> *From: Mara, hotel manager. By tonight I need the model for the booking system and an honest promise per 1,000 bookings. And tell me which model, and why.*

Data and costs as in CL1 and CL2: the offer costs **30 €**, a prevented cancellation is worth **100 €** (fictional), so we make the offer if p ≥ **0.3**. The bookings are split anew. Data: Antonio, Almeida, Nunes 2019, [Hotel booking demand datasets](https://doi.org/10.1016/j.dib.2018.11.126), CC BY 4.0, cleaned, a random sample.

## Setup

Run this cell once (about 10 seconds). You do not have to read the code.

In [ ]:
#@title Setup (run me) { display-mode: "form" }
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://fjoelsak.github.io/thm_wk-1609-predictiveanalytics/cl1-who-gets-the-offer/hotel-bookings.csv"
FILE = Path("hotel-bookings.csv")
df = pd.read_csv(FILE if FILE.exists() else URL)

# 60 % train, 20 % validation (for decisions), 20 % test (only for the last station)
train, rest = train_test_split(df, test_size=0.4, random_state=2, stratify=df["cancelled"])
valid, _test = train_test_split(rest, test_size=0.5, random_state=2, stratify=rest["cancelled"])

OFFER_COST, SAVED = 30, 100
CATEGORICAL = ["hotel", "month", "meal", "channel", "customer_type", "room_type"]
NUMERIC = [c for c in df.columns if c not in CATEGORICAL + ["cancelled", "parking_spaces"]]
ALL = NUMERIC + CATEGORICAL


def profit_per_1000(y, offered):
    """Profit per 1,000 bookings: +70 for each offer to a canceller, -30 for each other offer."""
    y, offered = np.asarray(y), np.asarray(offered, dtype=bool)
    return 1000 * np.mean(offered * (SAVED * y - OFFER_COST))


def report(y, offered):
    """Confusion matrix and measures for a list of offers."""
    tn, fp, fn, tp = confusion_matrix(y, offered, labels=[0, 1]).ravel()
    print(pd.DataFrame([[tn, fp], [fn, tp]], index=["came", "cancelled"], columns=["no offer", "offer"]), "\n")
    print(f"sensitivity {tp / (tp + fn):.2f}   specificity {tn / (tn + fp):.2f}   "
          f"precision {tp / (tp + fp) if tp + fp else float('nan'):.2f}   accuracy {(tp + tn) / len(y):.2f}")
    print(f"profit per 1,000 bookings: {profit_per_1000(y, offered):.0f} €")


def check(name, value, expected, tolerance=0.01):
    """Compare your value with the expected one."""
    ok = value is not None and abs(value - expected) <= tolerance
    print(f"{name}:", "correct ✅" if ok else "not yet ❌ (have another look at the output above)")


def hand_in(y, offered, decision, promise):
    """Last station: report on the test data and print the line to hand in."""
    report(y, offered)
    profit = profit_per_1000(y, offered)
    print(f"\nHAND IN >>> {decision} | promised {promise:.0f} € | reached {profit:.0f} € "
          f"| score {profit - abs(profit - promise):.0f}")


from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeClassifier, export_text


def fit_model(estimator, features=ALL, data=train, target=None):
    """Prepare the features (categories become 0/1 columns, no scaling) and train a model."""
    num = [f for f in features if f in NUMERIC]
    cat = [f for f in features if f in CATEGORICAL]
    prepare = make_column_transformer(("passthrough", num), (OneHotEncoder(drop="if_binary", sparse_output=False), cat))
    return make_pipeline(prepare, estimator).fit(data[features], data["cancelled"] if target is None else target)


def train_tree(features=ALL, **brakes):
    """One decision tree, e.g. train_tree(max_depth=4, min_samples_leaf=50)."""
    return fit_model(DecisionTreeClassifier(random_state=0, **brakes), features)


def train_forest(features=ALL, n_estimators=100, **settings):
    """Random forest, e.g. train_forest(n_estimators=300, min_samples_leaf=5)."""
    return fit_model(RandomForestClassifier(n_estimators=n_estimators, random_state=0, n_jobs=-1, **settings), features)


def train_boost(features=ALL, **settings):
    """Gradient boosting, e.g. train_boost(n_estimators=100, learning_rate=0.1, max_depth=3)."""
    return fit_model(GradientBoostingClassifier(random_state=0, **settings), features)


def train_logreg():
    """Logistic regression on all features, as in CL1."""
    prepare = make_column_transformer((StandardScaler(), NUMERIC), (OneHotEncoder(drop="first"), CATEGORICAL))
    return make_pipeline(prepare, LogisticRegression(max_iter=2000)).fit(train[ALL], train["cancelled"])


def proba(model, data):
    """Predicted probability of a cancellation."""
    return model.predict_proba(data[model.feature_names_in_])[:, 1]


def evaluate(model, data=valid, threshold=0.3):
    """AUC and profit per 1,000 bookings (offer if p >= threshold)."""
    p = proba(model, data)
    return roc_auc_score(data["cancelled"], p), profit_per_1000(data["cancelled"], p >= threshold)


def show(name, model, data=valid, threshold=0.3):
    """One line with the AUC and the profit of a model."""
    auc, profit = evaluate(model, data, threshold)
    print(f"{name:34s} AUC {auc:.3f}   profit {profit:6.0f} €")


def final_test(model, threshold, promise):
    """Station 4: evaluate the decision on the test data."""
    hand_in(_test["cancelled"], proba(model, _test) >= threshold, f"{type(model[-1]).__name__}, threshold {threshold}", promise)


print(f"{len(train)} train, {len(valid)} validation, {len(_test)} test bookings")
df.head()

## Station 1: Many trees

A forest grows many deep trees, each on a bootstrap sample of the bookings, and averages their probabilities. We compare it with logistic regression and with **one** tree of the same kind (`min_samples_leaf=5`).

**Predict:** will a forest of 100 trees beat logistic regression? By how much does it beat one tree (AUC and profit per 1,000 bookings)?

**Prediction:** *write here*

In [ ]:
show("logistic regression", train_logreg())
show("one tree (leaf of at least 5)", train_tree(min_samples_leaf=5))
show("forest, 100 trees", train_forest(min_samples_leaf=5))

How many trees are needed? The forest is the average of its trees, so we can look at the first *n* of them:

In [ ]:
forest = train_forest(n_estimators=300, min_samples_leaf=5)
x_valid = forest[0].transform(valid[ALL])
each_tree = np.array([t.predict_proba(x_valid)[:, 1] for t in forest[-1].estimators_])

sizes = [1, 2, 5, 10, 20, 50, 100, 200, 300]
plt.plot(sizes, [roc_auc_score(valid["cancelled"], each_tree[:n].mean(axis=0)) for n in sizes], color="green", linewidth=3, marker="o")
plt.xscale("log")
plt.xlabel("number of trees")
plt.ylabel("AUC on validation data")
plt.show()

**Change:** change the settings of the forest: `min_samples_leaf` (1, 5, 20), `max_features` (`"sqrt"`, `0.5`, `1.0`) or `max_depth`. Which combination earns most on the validation data?

In [ ]:
show("my forest", train_forest(min_samples_leaf=1, max_features="sqrt"))  # change me

**Read:** five trees vote on three bookings (1 = the tree says "cancels"). What does the code print?

```python
votes = np.array([[1, 0, 1],
                  [1, 1, 0],
                  [0, 0, 1],
                  [1, 0, 0],
                  [1, 0, 1]])
print(votes.mean(axis=0) >= 0.3)
```

**Answer:** *write here*

In [ ]:
votes = np.array([[1, 0, 1],
                  [1, 1, 0],
                  [0, 0, 1],
                  [1, 0, 0],
                  [1, 0, 1]])
print(votes.mean(axis=0) >= 0.3)

**Decide:** how many trees do you take, and why not 10,000?

**Answer:** *write here*

## Station 2: What does the forest look at?

**Predict:** which three features does the forest need most? Write them down before you run anything.

**Prediction:** *write here*

In [ ]:
forest = train_forest(min_samples_leaf=5)
shuffled = permutation_importance(forest, valid[ALL], valid["cancelled"], scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1)
importance = pd.Series(shuffled.importances_mean, index=ALL).sort_values()

importance.tail(8).plot.barh(color="green")
plt.xlabel("fall of the AUC if the column is shuffled")
plt.show()

`earlier_cancellations` counts bookings of the same guest that were created earlier and cancelled. According to the article about the data it is not clear that the cancellation happened **before** the current booking was made. If it did not, part of the feature would come from the future (compare `parking_spaces` in CL1).

**Change:** train the forest without `earlier_cancellations` and `earlier_stays`. How much do the AUC and the profit fall?

In [ ]:
features = [f for f in ALL if f not in ["earlier_cancellations", "earlier_stays"]]  # change me

show("forest with all features", train_forest(min_samples_leaf=5))
show("forest with fewer features", train_forest(features, min_samples_leaf=5))

**Read:** the AUC of a forest on the validation data is 0.78. After shuffling one column at a time it is as in the dictionary. Which feature does the forest need most?

```python
after = {"lead_time": 0.68, "price": 0.77, "channel": 0.70}
print(max(after, key=lambda feature: 0.78 - after[feature]))
```

**Answer:** *write here*

In [ ]:
after = {"lead_time": 0.68, "price": 0.77, "channel": 0.70}
print(max(after, key=lambda feature: 0.78 - after[feature]))

**Decide:** would you give `earlier_cancellations` to the model in the booking system? Give one reason.

**Answer:** *write here*

## Station 3: Boosting: how many rounds?

Boosting adds one small tree after the other; every tree corrects what the trees before still get wrong. Here every tree has 3 questions in a row (`max_depth=3`).

**Predict:** with the learning rate 0.1, will the AUC on the validation data rise up to 300 rounds or reach its best value earlier? And with the learning rate 0.5?

**Prediction:** *write here*

In [ ]:
def staged_auc(rate, rounds=300):
    """AUC on training and validation data after every round."""
    model = train_boost(n_estimators=rounds, learning_rate=rate)
    x_train, x_valid = model[0].transform(train[ALL]), model[0].transform(valid[ALL])
    on_train = [roc_auc_score(train["cancelled"], p[:, 1]) for p in model[-1].staged_predict_proba(x_train)]
    on_valid = [roc_auc_score(valid["cancelled"], p[:, 1]) for p in model[-1].staged_predict_proba(x_valid)]
    return np.array(on_train), np.array(on_valid)


plt.figure(figsize=(11, 4))
for i, rate in enumerate((0.1, 0.5)):
    on_train, on_valid = staged_auc(rate)
    plt.subplot(1, 2, i + 1)
    plt.plot(on_train, color="gray", linewidth=3, label="training")
    plt.plot(on_valid, color="green", linewidth=3, label="validation")
    plt.title(f"learning rate {rate}: best validation AUC after {on_valid.argmax() + 1} rounds")
    plt.xlabel("rounds")
    plt.ylabel("AUC")
    plt.legend()
plt.show()

**Change:** choose `n_estimators`, `learning_rate` and `max_depth` yourself and compare with the forest. Hint: a small learning rate needs many rounds.

In [ ]:
show("forest", train_forest(min_samples_leaf=5))
show("my boosting model", train_boost(n_estimators=100, learning_rate=0.1, max_depth=3))  # change me

**Read:** a booking was cancelled, so $y = 1$. Boosting starts with $F = 0.25$ and every tree predicts the residual $y - F$ exactly. What does the code print?

```python
y, F, nu = 1, 0.25, 0.5
for _ in range(2):
    F = F + nu * (y - F)
print(F)
```

**Answer:** *write here*

In [ ]:
y, F, nu = 1, 0.25, 0.5
for _ in range(2):
    F = F + nu * (y - F)
print(F)

**Decide:** how many rounds do you take with the learning rate 0.1? Give one reason.

**Answer:** *write here*

## Station 4: Promise and test

Choose your model, your threshold and **promise** Mara a profit per 1,000 bookings. `final_test` uses bookings that nobody has seen so far. **One shot.**

**Score = profit − |profit − promise|.**

In [ ]:
my_model = train_forest(min_samples_leaf=5)  # your decision: tree, forest, boosting or logistic regression
my_threshold = 0.5                           # your decision
my_promise = 0                               # your promise in € per 1,000 bookings

# final_test(my_model, my_threshold, my_promise)   # remove the first # when you are sure

**Hand in:** copy the line `HAND IN >>> ...` and add **one sentence**: which model should go live, and why?

**Answer:** *write here*

----
## Extra: for the fast ones

1. **Explain the forest with a tree:** train a tree of depth 3 on the *decisions of the forest* (offer yes or no at 0.3). How well does it copy the forest, and are the rules readable?
2. **XGBoost:** the same idea as `GradientBoostingClassifier`, faster. Try it (Colab has it installed).
3. **Exam question:** why does scaling `price` not change a random forest?

In [ ]:
# your space